# 00. EDA: 원천 데이터 탐색적 분석

**목적**: 민간 민원 상담 데이터셋(액티벤처, 엘지유플러스, 하나카드)의 원천데이터 및 라벨링데이터 구조·분포·품질을 파악하고, 이후 파이프라인(Pre-Init IDAS 요약 → 클러스터링) 설계에 필요한 통계를 산출한다.

**방법론**: zip 파일 내 JSON 파일을 파싱하여 pandas DataFrame으로 통합 후, 건수·길이·카테고리 분포·토큰 추정·비용 추정을 수행한다.

**산출물**: `data/processed/eda_summary.json` — EDA 핵심 통계를 JSON으로 저장하여 이후 노트북에서 참조

## 0. 환경 설정
Google Drive 마운트, 작업 디렉토리 이동, 필수 라이브러리 임포트

In [1]:
# === 환경 설정 ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/5stone_experiment/1_clustering_test')
print(f'작업 디렉토리: {os.getcwd()}')

import json
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive
작업 디렉토리: /content/drive/MyDrive/5stone_experiment/1_clustering_test


In [19]:
import unicodedata

## 0-1. 경로 설정 (PATH_CONFIG)
모든 파일 경로는 이 딕셔너리를 통해 참조한다.

In [2]:
# === 경로 설정 ===
BASE_DIR = Path('/content/drive/MyDrive/5stone_experiment/1_clustering_test')

PATH_CONFIG = {
    # 원천 데이터
    'raw_base': BASE_DIR / 'data' / 'raw' / '23.민간 민원 상담 LLM 사전학습 및 Instruction Tuning 데이터' / '3.개방데이터' / '1.데이터',
    # 가공 데이터
    'processed': BASE_DIR / 'data' / 'processed',
    # 중간 산출물
    'checkpoints': BASE_DIR / 'checkpoints',
    'embeddings': BASE_DIR / 'embeddings',
    'vector_db': BASE_DIR / 'vector_db',
    # 스크립트/노트북
    'scripts': BASE_DIR / 'scripts',
    'notebooks': BASE_DIR / 'notebooks',
}

# 원천/라벨링 데이터 하위 경로
PATH_CONFIG['train_source'] = PATH_CONFIG['raw_base'] / 'Training' / '01.원천데이터'
PATH_CONFIG['train_label'] = PATH_CONFIG['raw_base'] / 'Training' / '02.라벨링데이터'
PATH_CONFIG['val_source'] = PATH_CONFIG['raw_base'] / 'Validation' / '01.원천데이터'
PATH_CONFIG['val_label'] = PATH_CONFIG['raw_base'] / 'Validation' / '02.라벨링데이터'

# EDA 산출물
PATH_CONFIG['eda_summary'] = PATH_CONFIG['processed'] / 'eda_summary.json'

# 경로 존재 확인
for key, path in PATH_CONFIG.items():
    exists = path.exists()
    marker = '✓' if exists else '✗'
    print(f'  {marker} {key}: {path}')

  ✓ raw_base: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/raw/23.민간 민원 상담 LLM 사전학습 및 Instruction Tuning 데이터/3.개방데이터/1.데이터
  ✓ processed: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed
  ✓ checkpoints: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints
  ✓ embeddings: /content/drive/MyDrive/5stone_experiment/1_clustering_test/embeddings
  ✓ vector_db: /content/drive/MyDrive/5stone_experiment/1_clustering_test/vector_db
  ✓ scripts: /content/drive/MyDrive/5stone_experiment/1_clustering_test/scripts
  ✓ notebooks: /content/drive/MyDrive/5stone_experiment/1_clustering_test/notebooks
  ✓ train_source: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/raw/23.민간 민원 상담 LLM 사전학습 및 Instruction Tuning 데이터/3.개방데이터/1.데이터/Training/01.원천데이터
  ✓ train_label: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/raw/23.민간 민원 상담 LLM 사전학습 및 Instruction Tuning 데이터/3.개방데이터/1.데이터/Training/02.라벨링데이터
  ✓ val_source: /co

## 0-2. 유틸리티 함수 정의
zip 파일 내 JSON 파싱, 토큰 수 추정 등 공통 함수

In [3]:
# === 유틸리티 함수 ===

def load_json_from_zip(zip_path: Path) -> list:
    """
    zip 파일 내 모든 JSON 파일을 읽어 레코드 리스트로 반환한다.
    각 JSON 파일은 리스트 형태([{...}, ...])를 가정한다.
    """
    records = []
    with zipfile.ZipFile(zip_path, 'r') as zf:
        json_files = [f for f in zf.namelist()
                      if f.endswith('.json') and not f.startswith('__MACOSX')]
        for jf in tqdm(json_files, desc=f'{zip_path.name}', leave=False):
            with zf.open(jf) as fp:
                data = json.loads(fp.read().decode('utf-8'))
                if isinstance(data, list):
                    records.extend(data)
                else:
                    records.append(data)
    return records


def estimate_tokens_korean(text: str) -> int:
    """
    한국어 텍스트의 토큰 수를 근사 추정한다.
    Gemini 토크나이저 기준, 한국어는 대략 글자 수의 1.0~1.5배 토큰.
    보수적으로 글자 수 × 1.3 을 사용한다.
    """
    return int(len(text) * 1.3)


def extract_source_df(records: list) -> pd.DataFrame:
    """
    원천데이터 레코드 리스트를 DataFrame으로 변환한다.
    """
    rows = []
    for r in records:
        content = r.get('consulting_content', '')
        rows.append({
            'source_id': r.get('source_id', ''),
            'source': r.get('source', ''),
            'consulting_category': r.get('consulting_category', ''),
            'client_gender': r.get('client_gender', ''),
            'client_age': r.get('client_age', ''),
            'consulting_turns': int(r.get('consulting_turns', 0)),
            'consulting_length': int(r.get('consulting_length', 0)),
            'content_char_len': len(content),
            'content_token_est': estimate_tokens_korean(content),
        })
    return pd.DataFrame(rows)


def extract_label_df(records: list, label_type: str) -> pd.DataFrame:
    """
    라벨링데이터 레코드 리스트를 DataFrame으로 변환한다.
    label_type: '분류', '요약', '질의응답'
    """
    rows = []
    for r in records:
        for inst_group in r.get('instructions', []):
            for d in inst_group.get('data', []):
                rows.append({
                    'source_id': r.get('source_id', ''),
                    'source': r.get('source', ''),
                    'consulting_category': r.get('consulting_category', ''),
                    'label_type': label_type,
                    'task_category': d.get('task_category', ''),
                    'instruction': d.get('instruction', ''),
                    'output': d.get('output', ''),
                    'output_char_len': len(d.get('output', '')),
                })
    return pd.DataFrame(rows)


print('유틸리티 함수 정의 완료')

유틸리티 함수 정의 완료


## 1. 원천데이터 로드 및 기본 통계
Training + Validation 원천데이터 zip 파일을 파싱하여 전체 건수, 출처별 건수, 상담 길이 분포를 파악한다.

**체크포인트**: `checkpoints/eda_source_df.parquet`가 존재하면 로드, 없으면 zip에서 새로 파싱

In [4]:
# === 원천데이터 로드 ===

def load_all_source_data() -> pd.DataFrame:
    """
    Training + Validation 원천데이터를 모두 로드하여 하나의 DataFrame으로 반환.
    체크포인트가 존재하면 캐시에서 로드한다.
    """
    ckpt_path = PATH_CONFIG['checkpoints'] / 'eda_source_df.parquet'

    if ckpt_path.exists():
        print(f'체크포인트 로드: {ckpt_path}')
        return pd.read_parquet(ckpt_path)

    source_zips = {
        'train': sorted(PATH_CONFIG['train_source'].glob('TS_*.zip')),
        'val': sorted(PATH_CONFIG['val_source'].glob('VS_*.zip')),
    }

    all_dfs = []
    for split, zips in source_zips.items():
        print(f'\n--- {split.upper()} 원천데이터 ---')
        for zp in zips:
            print(f'  로드 중: {zp.name}')
            records = load_json_from_zip(zp)
            df = extract_source_df(records)
            df['split'] = split
            df['zip_file'] = zp.name
            all_dfs.append(df)
            print(f'    → {len(df):,}건')

    combined = pd.concat(all_dfs, ignore_index=True)

    # 체크포인트 저장
    combined.to_parquet(ckpt_path, index=False)
    print(f'\n체크포인트 저장: {ckpt_path}')
    print(f'총 원천데이터: {len(combined):,}건')

    return combined


source_df = load_all_source_data()
source_df.head()


--- TRAIN 원천데이터 ---
  로드 중: TS_액티벤처.zip


TS_액티벤처.zip:   0%|          | 0/800 [00:00<?, ?it/s]

    → 800건
  로드 중: TS_엘지유플러스.zip


TS_엘지유플러스.zip:   0%|          | 0/3216 [00:00<?, ?it/s]

    → 3,216건
  로드 중: TS_하나카드.zip


...

VS_엘지유플러스.zip:   0%|          | 0/384 [00:00<?, ?it/s]

    → 384건
  로드 중: VS_하나카드.zip


VS_하나카드.zip:   0%|          | 0/776 [00:00<?, ?it/s]

    → 776건

체크포인트 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints/eda_source_df.parquet
총 원천데이터: 11,033건


,source_id,source,consulting_category,client_gender,client_age,consulting_turns,consulting_length,content_char_len,content_token_est,split,zip_file
0,90100,액티벤처,상품예약 및 결제,,,8,244,1109,1441,train,TS_액티벤처.zip
1,90010,액티벤처,상품예약 및 결제,,,10,147,679,882,train,TS_액티벤처.zip
2,90102,액티벤처,상품예약 및 결제,,,10,170,760,988,train,TS_액티벤처.zip
3,90108,액티벤처,요금 및 견적,,,10,203,907,1179,train,TS_액티벤처.zip
4,90101,액티벤처,요금 및 견적,,,10,177,818,1063,train,TS_액티벤처.zip


## 2. 원천데이터 기본 통계
출처별 건수, split별 건수, 상담 길이(글자/토큰) 분포 확인

In [5]:
# === 원천데이터 기본 통계 ===

def print_source_stats(df: pd.DataFrame) -> dict:
    """
    원천데이터 DataFrame의 기본 통계를 출력하고, 핵심 수치를 dict로 반환한다.
    """
    stats = {}

    # 1) 전체 건수
    print(f'=== 전체 건수: {len(df):,} ===')
    stats['total_count'] = len(df)

    # 2) split별 건수
    print(f'\n--- split별 ---')
    split_counts = df['split'].value_counts().to_dict()
    for k, v in split_counts.items():
        print(f'  {k}: {v:,}')
    stats['split_counts'] = split_counts

    # 3) 출처별 건수
    print(f'\n--- 출처(source)별 ---')
    source_counts = df['source'].value_counts().to_dict()
    for k, v in source_counts.items():
        print(f'  {k}: {v:,}')
    stats['source_counts'] = source_counts

    # 4) 출처 × split 크로스탭
    print(f'\n--- 출처 × split ---')
    ct = pd.crosstab(df['source'], df['split'], margins=True)
    print(ct.to_string())

    # 5) 상담 길이 분포 (글자 수 기준)
    print(f'\n--- 상담 content 길이 (글자 수) ---')
    length_desc = df['content_char_len'].describe(percentiles=[.25, .5, .75, .9, .95, .99])
    print(length_desc.to_string())
    stats['content_char_len'] = length_desc.to_dict()

    # 6) 토큰 수 추정 분포
    print(f'\n--- 토큰 수 추정 (×1.3) ---')
    token_desc = df['content_token_est'].describe(percentiles=[.25, .5, .75, .9, .95, .99])
    print(token_desc.to_string())
    stats['content_token_est'] = token_desc.to_dict()

    # 7) 총 토큰 수 추정
    total_tokens = df['content_token_est'].sum()
    print(f'\n총 추정 토큰 수: {total_tokens:,.0f}')
    stats['total_tokens_est'] = int(total_tokens)

    # 8) consulting_turns 분포
    print(f'\n--- 상담 턴 수 ---')
    turn_desc = df['consulting_turns'].describe()
    print(turn_desc.to_string())
    stats['consulting_turns'] = turn_desc.to_dict()

    return stats


source_stats = print_source_stats(source_df)

=== 전체 건수: 11,033 ===

--- split별 ---
  train: 9,773
  val: 1,260

--- 출처(source)별 ---
  하나카드: 6,533
  엘지유플러스: 3,600
  액티벤처: 900

--- 출처 × split ---
split   train   val    All
source                    
액티벤처      800   100    900
엘지유플러스   3216   384   3600
하나카드     5757   776   6533
All      9773  1260  11033

--- 상담 content 길이 (글자 수) ---
count    11033.000000
mean      1330.395722
std        380.538186
min        590.000000
25%       1023.000000
50%       1306.000000
75%       1591.000000
90%       1818.000000
95%       1978.400000
99%       2308.040000
max       5573.000000

--- 토큰 수 추정 (×1.3) ---
count    11033.000000
mean      1729.066165
std        494.695220
min        767.000000
25%       1329.000000
50%       1697.000000
75%       2068.000000
90%       2363.000000
95%       2571.400000
99%       2999.720000
max       7244.000000

총 추정 토큰 수: 19,076,787

--- 상담 턴 수 ---
count    11033.000000
mean        37.708964
std         14.198308
min          2.000000
25%         29.000000
50

## 3. 상담 카테고리(consulting_category) 분포
원천데이터의 `consulting_category` 필드 분포를 출처별로 확인한다. 이후 클러스터링 결과의 coarse-label 평가 기준으로 활용할 예정.

In [6]:
# === 카테고리 분포 ===

def analyze_categories(df: pd.DataFrame) -> dict:
    """
    consulting_category 분포를 전체 및 출처별로 분석한다.
    """
    cat_stats = {}

    # 전체 카테고리 분포
    print('=== 전체 카테고리 분포 ===')
    cat_counts = df['consulting_category'].value_counts()
    cat_pct = df['consulting_category'].value_counts(normalize=True) * 100
    cat_table = pd.DataFrame({'건수': cat_counts, '비율(%)': cat_pct.round(1)})
    print(cat_table.to_string())
    print(f'\n고유 카테고리 수: {df["consulting_category"].nunique()}')
    cat_stats['total'] = cat_counts.to_dict()

    # 출처별 카테고리 분포
    for src in sorted(df['source'].unique()):
        print(f'\n--- {src} ---')
        sub = df[df['source'] == src]
        sub_counts = sub['consulting_category'].value_counts()
        sub_pct = sub['consulting_category'].value_counts(normalize=True) * 100
        sub_table = pd.DataFrame({'건수': sub_counts, '비율(%)': sub_pct.round(1)})
        print(sub_table.to_string())
        print(f'  고유 카테고리 수: {sub["consulting_category"].nunique()}')
        cat_stats[src] = sub_counts.to_dict()

    return cat_stats


category_stats = analyze_categories(source_df)

=== 전체 카테고리 분포 ===
                            건수  비율(%)
consulting_category                  
선결제/즉시출금                   927    8.4
이용내역 안내                    919    8.3
요금 안내                      694    6.3
요금 납부                      544    4.9
요금제 변경                     493    4.5
상품예약 및 결제                  477    4.3
한도상향 접수/처리                 402    3.6
도난/분실 신청/해제                398    3.6
요금 및 견적                    382    3.5
선택약정 할인                    380    3.4
결제대금 안내                    331    3.0
납부 방법 변경                   317    2.9
승인취소/매출취소 안내               301    2.7
이벤트 안내                     223    2.0
부가서비스 안내                   201    1.8
소액 결제                      179    1.6
정부지원 바우처 (등유, 임신 등)        167    1.5
연체대금 즉시출금                  163    1.5
한도 안내                      163    1.5
결제계좌 안내/변경                 161    1.5
이용방법 안내                    161    1.5
일부결제대금이월약정 해지              161    1.5
휴대폰 정지/분실/파손               160    1.5
결제일 안내/변경/취소               150 

## 4. 라벨링데이터 로드 및 분석
Training + Validation 라벨링데이터(분류, 요약, 질의응답)의 건수와 구조를 파악한다.

**체크포인트**: `checkpoints/eda_label_df.parquet`

In [21]:
# === 라벨링데이터 로드 ===

def load_all_label_data() -> pd.DataFrame:
    """
    Training + Validation 라벨링데이터를 모두 로드.
    파일명 패턴: TL_{출처}_{유형}.zip, VL_{출처}_{유형}.zip
    유형: 분류, 요약, 질의응답
    """
    ckpt_path = PATH_CONFIG['checkpoints'] / 'eda_label_df.parquet'

    if ckpt_path.exists():
        print(f'체크포인트 로드: {ckpt_path}')
        return pd.read_parquet(ckpt_path)

    label_dirs = {
        'train': PATH_CONFIG['train_label'],
        'val': PATH_CONFIG['val_label'],
    }

    # 유형 매핑: 파일명 키워드 → label_type
    type_map = {'분류': '분류', '요약': '요약', '질의응답': '질의응답'}

    all_dfs = []
    for split, label_dir in label_dirs.items():
        print(f'\n--- {split.upper()} 라벨링데이터 ---')
        for zp in sorted(label_dir.glob('*.zip')):
            # 파일명에서 유형 추출
            normalized_name = unicodedata.normalize('NFC', zp.name)
            label_type = None
            for key in type_map:
                print(key + "," + zp.name)
                if key in normalized_name:
                    label_type = type_map[key]
                    break
            if label_type is None:
                print(f'  [SKIP] 유형 불명: {zp.name}')
                continue

            print(f'  로드 중: {zp.name} (유형: {label_type})')
            records = load_json_from_zip(zp)
            df = extract_label_df(records, label_type)
            df['split'] = split
            df['zip_file'] = zp.name
            all_dfs.append(df)
            print(f'    → {len(df):,}건 (instruction-output 쌍)')

    combined = pd.concat(all_dfs, ignore_index=True)

    combined.to_parquet(ckpt_path, index=False)
    print(f'\n체크포인트 저장: {ckpt_path}')
    print(f'총 라벨링데이터: {len(combined):,}건')

    return combined


label_df = load_all_label_data()
label_df.head()


--- TRAIN 라벨링데이터 ---
분류,TL_액티벤처_분류.zip
  로드 중: TL_액티벤처_분류.zip (유형: 분류)


TL_액티벤처_분류.zip:   0%|          | 0/4000 [00:00<?, ?it/s]

    → 4,000건 (instruction-output 쌍)
분류,TL_액티벤처_요약.zip
요약,TL_액티벤처_요약.zip
  로드 중: TL_액티벤처_요약.zip (유형: 요약)


TL_액티벤처_요약.zip:   0%|          | 0/3164 [00:00<?, ?it/s]

    → 3,164건 (instruction-output 쌍)
분류,TL_액티벤처_질의응답.zip
요약,TL_액티벤처_질의응답.zip
질의응답,TL_액티벤처_질의응답.zip
  로드 중: TL_액티벤처_질의응답.zip (유형: 질의응답)


...

VL_하나카드_요약.zip:   0%|          | 0/2112 [00:00<?, ?it/s]

    → 2,112건 (instruction-output 쌍)
분류,VL_하나카드_질의응답.zip
요약,VL_하나카드_질의응답.zip
질의응답,VL_하나카드_질의응답.zip
  로드 중: VL_하나카드_질의응답.zip (유형: 질의응답)


VL_하나카드_질의응답.zip:   0%|          | 0/1923 [00:00<?, ?it/s]

    → 1,923건 (instruction-output 쌍)

체크포인트 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints/eda_label_df.parquet
총 라벨링데이터: 119,182건


,source_id,source,consulting_category,label_type,task_category,instruction,output,output_char_len,split,zip_file
0,90004,액티벤처,상품예약 및 결제,분류,상담 주제,"상담 주제는 ""상품 및 서비스 일반"", ""주문/결제/입금 확인"", ""취소/반품/교환...",상품 및 서비스 일반,11,train,TL_액티벤처_분류.zip
1,90005,액티벤처,상품예약 및 결제,분류,상담 결과,"이 민원의 상담 결과는 ""만족"", ""미흡"", ""해결 불가"", ""추가 상담 필요"" 중...",만족,2,train,TL_액티벤처_분류.zip
2,90002,액티벤처,요금 및 견적,분류,상담 사유,"이번 상담이 이루어진 원인은 ""업체""에 있니, 아니면 ""민원인""에게 있니?",민원인,3,train,TL_액티벤처_분류.zip
3,90001,액티벤처,요금 및 견적,분류,상담 주제,"이 상담 내용은 ""상품 및 서비스 일반"", ""주문/결제/입금 확인"", ""취소/반품/...",상품 및 서비스 일반,11,train,TL_액티벤처_분류.zip
4,90001,액티벤처,요금 및 견적,분류,상담 요건,"해당 상담 내용이 ""단일 요건 민원"", ""다수 요건 민원"" 중 어디에 속하는지 알려줘.",단일 요건 민원,8,train,TL_액티벤처_분류.zip


## 5. 라벨링데이터 통계
유형별(분류/요약/질의응답) 건수, 출처별 건수, 분류 라벨링의 output 분포(9개 카테고리 기준) 확인

In [22]:
# === 라벨링데이터 통계 ===

def print_label_stats(df: pd.DataFrame) -> dict:
    """
    라벨링데이터 통계를 출력하고 핵심 수치를 반환한다.
    """
    stats = {}

    print(f'=== 총 라벨링데이터: {len(df):,}건 ===')

    # 유형별 건수
    print(f'\n--- 유형별 ---')
    type_counts = df['label_type'].value_counts().to_dict()
    for k, v in type_counts.items():
        print(f'  {k}: {v:,}')
    stats['type_counts'] = type_counts

    # 유형 × 출처 크로스탭
    print(f'\n--- 유형 × 출처 ---')
    ct = pd.crosstab(df['label_type'], df['source'], margins=True)
    print(ct.to_string())

    # 분류 라벨링의 output 분포 (ground truth 카테고리)
    cls_df = df[df['label_type'] == '분류']
    if len(cls_df) > 0:
        print(f'\n--- 분류 라벨링: output 분포 (ground truth) ---')
        cls_output = cls_df['output'].value_counts()
        cls_pct = cls_df['output'].value_counts(normalize=True) * 100
        cls_table = pd.DataFrame({'건수': cls_output, '비율(%)': cls_pct.round(1)})
        print(cls_table.to_string())
        stats['classification_labels'] = cls_output.to_dict()

    # 분류 라벨링의 task_category 분포
    if len(cls_df) > 0:
        print(f'\n--- 분류 라벨링: task_category 분포 ---')
        tc_counts = cls_df['task_category'].value_counts()
        print(tc_counts.to_string())
        stats['classification_task_categories'] = tc_counts.to_dict()

    # 요약 라벨링의 output 길이 분포
    summ_df = df[df['label_type'] == '요약']
    if len(summ_df) > 0:
        print(f'\n--- 요약 라벨링: output 길이 (글자 수) ---')
        print(summ_df['output_char_len'].describe().to_string())
        stats['summary_output_len'] = summ_df['output_char_len'].describe().to_dict()

    return stats


label_stats = print_label_stats(label_df)

=== 총 라벨링데이터: 119,182건 ===

--- 유형별 ---
  분류: 55,185
  요약: 35,848
  질의응답: 28,149

--- 유형 × 출처 ---
source       액티벤처  엘지유플러스   하나카드     All
label_type                              
분류           4500   18000  32685   55185
요약           3537   13096  19215   35848
질의응답         2463    8324  17362   28149
All         10500   39420  69262  119182

--- 분류 라벨링: output 분포 (ground truth) ---
                   건수  비율(%)
output                      
민원인             10642   19.3
단일 요건 민원         9239   16.7
만족               8995   16.3
업무 처리 상담         5559   10.1
일반 문의 상담         5450    9.9
주문/결제/입금확인       3442    6.2
상품 및 서비스 일반      3195    5.8
다수 요건 민원         1794    3.3
회원 관리            1513    2.7
취소/반품/교환/환불/AS   1399    2.5
추가 상담 필요          940    1.7
미흡                876    1.6
이벤트/할인            761    1.4
제휴                584    1.1
업체                391    0.7
해결 불가             222    0.4
기타                115    0.2
배송 문의              34    0.1
고충 상담              24    0.0
콘텐츠  

## 6. 복수 의도 상담 비율 추정
상담 대화 내 고객 발화에서 화제 전환 패턴("또 하나는", "그리고", "한 가지 더" 등)을 탐지하여, 복수 의도를 포함하는 상담의 비율을 추정한다. Pre-Init에서 의도 단위 분리가 필요한 규모를 가늠하기 위함.

In [23]:
# === 복수 의도 추정 ===
import re

def estimate_multi_intent_ratio(df: pd.DataFrame) -> dict:
    """
    원천데이터에서 복수 의도를 포함할 가능성이 있는 상담 비율을 추정한다.
    체크포인트에서 원본 content를 다시 읽어야 하므로, zip에서 직접 로드한다.
    """
    ckpt_path = PATH_CONFIG['checkpoints'] / 'eda_multi_intent.json'

    if ckpt_path.exists():
        print(f'체크포인트 로드: {ckpt_path}')
        with open(ckpt_path, 'r') as f:
            return json.load(f)

    # 화제 전환 패턴 (고객 발화 기준)
    topic_switch_patterns = [
        r'또\s*하나',
        r'한\s*가지\s*더',
        r'그리고.*여쭤',
        r'다른\s*건',
        r'추가로',
        r'별도로',
        r'또\s*다른',
        r'하나\s*더',
        r'마지막으로',
        r'그\s*외에',
    ]
    combined_pattern = re.compile('|'.join(topic_switch_patterns))

    # zip에서 content 로드 (원천데이터만)
    multi_count = 0
    total_count = 0
    switch_counts = Counter()  # 패턴별 매칭 횟수

    for split_key, dir_key in [('train', 'train_source'), ('val', 'val_source')]:
        for zp in sorted(PATH_CONFIG[dir_key].glob('*.zip')):
            records = load_json_from_zip(zp)
            for r in records:
                content = r.get('consulting_content', '')
                # 고객 발화만 추출
                customer_lines = []
                for line in content.split('\n'):
                    if line.startswith('고객:') or line.startswith('손님:'):
                        customer_lines.append(line)
                customer_text = '\n'.join(customer_lines)

                total_count += 1
                matches = combined_pattern.findall(customer_text)
                if matches:
                    multi_count += 1
                    for m in matches:
                        switch_counts[m.strip()] += 1

    result = {
        'total_count': total_count,
        'multi_intent_count': multi_count,
        'multi_intent_ratio': round(multi_count / total_count * 100, 1) if total_count > 0 else 0,
        'top_patterns': dict(switch_counts.most_common(10)),
    }

    print(f'총 상담: {total_count:,}')
    print(f'복수 의도 추정: {multi_count:,} ({result["multi_intent_ratio"]}%)')
    print(f'\n상위 화제 전환 패턴:')
    for pat, cnt in switch_counts.most_common(10):
        print(f'  "{pat}": {cnt:,}')

    with open(ckpt_path, 'w') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f'\n체크포인트 저장: {ckpt_path}')

    return result


multi_intent_stats = estimate_multi_intent_ratio(source_df)

TS_액티벤처.zip:   0%|          | 0/800 [00:00<?, ?it/s]

TS_엘지유플러스.zip:   0%|          | 0/3216 [00:00<?, ?it/s]

TS_하나카드.zip:   0%|          | 0/5757 [00:00<?, ?it/s]

VS_액티벤처.zip:   0%|          | 0/100 [00:00<?, ?it/s]

VS_엘지유플러스.zip:   0%|          | 0/384 [00:00<?, ?it/s]

VS_하나카드.zip:   0%|          | 0/776 [00:00<?, ?it/s]

총 상담: 11,033
복수 의도 추정: 776 (7.0%)

상위 화제 전환 패턴:
  "마지막으로": 179
  "하나 더": 140
  "추가로": 123
  "별도로": 81
  "다른 건": 68
  "또 하나": 66
  "또 다른": 48
  "한 가지 더": 41
  "그 외에": 19
  "그리고 하나 더 여쭤": 7

체크포인트 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints/eda_multi_intent.json


## 7. Gemini API 비용 추정
Pre-Init 단계(IDAS 의도 분리·요약)에서 전체 원천데이터를 Gemini flash-lite로 처리할 때의 예상 비용을 산출한다.

**기준**: gemini-2.0-flash-lite 가격 (2025.03 기준)
- Input: $0.075 / 1M tokens
- Output: $0.30 / 1M tokens

In [24]:
# === Gemini API 비용 추정 ===

def estimate_gemini_cost(source_stats: dict, exchange_rate: float = 1450.0) -> dict:
    """
    Pre-Init (IDAS 요약) 단계의 Gemini API 비용을 추정한다.

    가정:
    - 모델: gemini-2.0-flash-lite
    - Input = 시스템 프롬프트(~500 토큰) + consulting_content
    - Output = 의도별 요약 (평균 ~200 토큰 추정)
    """
    # 가격 (USD per 1M tokens)
    input_price = 0.075   # $/1M input tokens
    output_price = 0.30   # $/1M output tokens

    total_count = source_stats['total_count']
    total_input_tokens = source_stats['total_tokens_est']

    # 시스템 프롬프트 오버헤드 (건당 ~500 토큰)
    system_prompt_tokens = total_count * 500
    total_input_with_prompt = total_input_tokens + system_prompt_tokens

    # Output 추정: 건당 평균 200 토큰
    total_output_tokens = total_count * 200

    # 비용 계산
    input_cost_usd = (total_input_with_prompt / 1_000_000) * input_price
    output_cost_usd = (total_output_tokens / 1_000_000) * output_price
    total_cost_usd = input_cost_usd + output_cost_usd
    total_cost_krw = total_cost_usd * exchange_rate

    result = {
        'total_input_tokens': total_input_with_prompt,
        'total_output_tokens': total_output_tokens,
        'input_cost_usd': round(input_cost_usd, 4),
        'output_cost_usd': round(output_cost_usd, 4),
        'total_cost_usd': round(total_cost_usd, 4),
        'total_cost_krw': round(total_cost_krw, 0),
        'exchange_rate': exchange_rate,
        'budget_krw': 50000,
        'budget_remaining_krw': round(50000 - total_cost_krw, 0),
    }

    print('=== Pre-Init (IDAS 요약) Gemini API 비용 추정 ===')
    print(f'  모델: gemini-2.0-flash-lite')
    print(f'  총 상담 건수: {total_count:,}')
    print(f'  총 입력 토큰 (프롬프트 포함): {total_input_with_prompt:,.0f}')
    print(f'  총 출력 토큰 추정: {total_output_tokens:,.0f}')
    print(f'  ---')
    print(f'  입력 비용: ${input_cost_usd:.4f}')
    print(f'  출력 비용: ${output_cost_usd:.4f}')
    print(f'  합계: ${total_cost_usd:.4f} (≈ ₩{total_cost_krw:,.0f})')
    print(f'  ---')
    print(f'  예산: ₩50,000')
    print(f'  Pre-Init 후 잔여 예산: ₩{result["budget_remaining_krw"]:,.0f}')

    return result


cost_stats = estimate_gemini_cost(source_stats)

=== Pre-Init (IDAS 요약) Gemini API 비용 추정 ===
  모델: gemini-2.0-flash-lite
  총 상담 건수: 11,033
  총 입력 토큰 (프롬프트 포함): 24,593,287
  총 출력 토큰 추정: 2,206,600
  ---
  입력 비용: $1.8445
  출력 비용: $0.6620
  합계: $2.5065 (≈ ₩3,634)
  ---
  예산: ₩50,000
  Pre-Init 후 잔여 예산: ₩46,366


## 8. 인구통계 필드 분포 (참고)
client_gender, client_age 필드의 채워진 비율과 분포를 확인한다. 파이프라인에 직접 사용하지는 않지만, 데이터 품질 점검 목적.

In [25]:
# === 인구통계 필드 분포 ===

def analyze_demographics(df: pd.DataFrame) -> dict:
    """
    client_gender, client_age 필드의 채워진 비율과 분포를 분석한다.
    """
    demo_stats = {}

    for col in ['client_gender', 'client_age']:
        filled = df[col].replace('', pd.NA).notna().sum()
        fill_rate = filled / len(df) * 100
        print(f'--- {col} ---')
        print(f'  채워진 건수: {filled:,} / {len(df):,} ({fill_rate:.1f}%)')

        if filled > 0:
            dist = df[df[col].replace('', pd.NA).notna()][col].value_counts()
            print(dist.to_string())
            demo_stats[col] = {
                'fill_rate': round(fill_rate, 1),
                'distribution': dist.to_dict(),
            }
        else:
            demo_stats[col] = {'fill_rate': 0, 'distribution': {}}
        print()

    return demo_stats


demo_stats = analyze_demographics(source_df)

--- client_gender ---
  채워진 건수: 6,533 / 11,033 (59.2%)
client_gender
여자    3560
남자    2973

--- client_age ---
  채워진 건수: 6,533 / 11,033 (59.2%)
client_age
50대    1848
40대    1627
60대    1409
30대     824
70대     421
20대     375
10대      29



## 9. EDA 요약 저장
모든 핵심 통계를 `data/processed/eda_summary.json`에 저장하여, 이후 노트북에서 참조할 수 있도록 한다.

In [26]:
# === EDA 요약 저장 ===

def save_eda_summary(
    source_stats: dict,
    category_stats: dict,
    label_stats: dict,
    multi_intent_stats: dict,
    cost_stats: dict,
    demo_stats: dict,
) -> None:
    """
    EDA 핵심 통계를 JSON 파일로 저장한다.
    """
    summary = {
        'source_data': source_stats,
        'categories': category_stats,
        'label_data': label_stats,
        'multi_intent': multi_intent_stats,
        'cost_estimation': cost_stats,
        'demographics': demo_stats,
    }

    output_path = PATH_CONFIG['eda_summary']
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2, default=str)

    print(f'EDA 요약 저장 완료: {output_path}')
    print(f'파일 크기: {output_path.stat().st_size / 1024:.1f} KB')


save_eda_summary(
    source_stats=source_stats,
    category_stats=category_stats,
    label_stats=label_stats,
    multi_intent_stats=multi_intent_stats,
    cost_stats=cost_stats,
    demo_stats=demo_stats,
)

EDA 요약 저장 완료: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/eda_summary.json
파일 크기: 12.0 KB


## 10. 샘플 상담 확인
각 출처에서 1건씩 원본 상담 내용을 출력하여 데이터 품질과 형태를 육안으로 확인한다.

In [27]:
# === 샘플 상담 출력 ===

def print_sample_consultations(n_per_source: int = 1) -> None:
    """
    각 출처별로 n건의 원본 상담 내용을 출력한다.
    zip에서 직접 로드하여 consulting_content 원문을 보여준다.
    """
    for split_key, dir_key in [('train', 'train_source')]:
        for zp in sorted(PATH_CONFIG[dir_key].glob('*.zip')):
            print(f'\n{"="*60}')
            print(f'📂 {zp.name}')
            print(f'{"="*60}')

            with zipfile.ZipFile(zp, 'r') as zf:
                json_files = [f for f in zf.namelist()
                              if f.endswith('.json') and not f.startswith('__MACOSX')]
                # 첫 번째 JSON 파일에서 n건만 확인
                if json_files:
                    with zf.open(json_files[0]) as fp:
                        data = json.loads(fp.read().decode('utf-8'))
                        samples = data[:n_per_source] if isinstance(data, list) else [data]

                        for i, r in enumerate(samples):
                            print(f'\n--- 샘플 {i+1} ---')
                            print(f'source_id: {r.get("source_id")}')
                            print(f'category: {r.get("consulting_category")}')
                            print(f'turns: {r.get("consulting_turns")}')
                            content = r.get('consulting_content', '')
                            # 처음 500자만 출력
                            if len(content) > 500:
                                print(f'content (처음 500자):\n{content[:500]}...')
                            else:
                                print(f'content:\n{content}')


print_sample_consultations(n_per_source=1)


📂 TS_액티벤처.zip

--- 샘플 1 ---
source_id: 90100
category: 상품예약 및 결제
turns: 8
content (처음 500자):
고객: 안녕하세요, ▲▲월 ▲▲일 출발로 5일 일정으로 2명 예약 가능한지 궁금해요. 리조트는 내일 전화로 문의할 예정이에요. 
비치가 좋은 곳으로, 쿠타 쪽으로 가는 택시가 힘들지 않았으면 좋겠어요. 우붓이나 절벽 위의 메리디안은 제외하고 싶고요. 
호텔은 그랜드 하얏트나 미라지를 생각하고 있어요. 자매 두 명이라 풀빌라를 꼭 가고 싶진 않지만, 발리니까 풀빌라도 괜찮을 것 같아요. 
3박이라 힘들겠지만, 가능하다면 두 가지 호텔에서 지내보고 싶어요. 추천해주실 만한 곳이 있으면 알려주세요. 
담당자 직통 전화번호도 부탁드려요. 내일 좀 더 알아보고 전화드릴게요.
상담사: 안녕하세요, 고객님! ▲▲▲▲입니다. 주말 잘 보내셨나요? ▲▲월 ▲▲일 오전 출발로 3박 5일 일정은 예약 가능해요. 비치가 예쁜 곳으로는 누사두아나 베노아 지역의 호텔이 적합할 것 같아요. 그랜드 하얏트와 그랜드 미라지 모두 괜찮은 선택이에요. 두 곳 모두 쿠타 지역으로 이동하기에 편리하고, 차량으로 20-3...

📂 TS_엘지유플러스.zip

--- 샘플 1 ---
source_id: 41043
category: 가입 안내
turns: 39
content (처음 500자):
상담사: 안녕하세요? 일상의 즐거운 변화 LGU+ <NAME>입니다.
고객: 네, 안녕하세요? 여기 프리미어요금제 약정할인 신청할려구요.
상담사: 아 네, 연락 잘 주셨구요, 저희가 확인해 보고 조회해 드리겠습니다. 입력하신 번호로 지금 <NAME> 고객님 본인되실까요?
고객: 네.
상담사: 아 네, 확인 감사합니다. 그 프리미어 약정할인이 요금제 2년 동안 가입이 되시구요. <CHARGE>씩 받는 건데요.
고객: 네.
상담사: 그러시면 동의만 해주시면 되는데 끊지 마시고 잠시만 기다려 주시겠습니까?
고객: 네, 많이 기다